# Visualize CLASP Embedding

Run and compare three visualization paths for the same dataset: the benchmark-derived default preset, the optimized parameters if they exist, and an editable parameter block for manual experiments.


In [ ]:
from importlib import reload

import matplotlib.pyplot as plt
import pandas as pd

import clasp.notebook_utils as notebook_utils
from clasp import ClaspEstimator, available_presets, get_preset

notebook_utils = reload(notebook_utils)


In [ ]:
selected_dataset = "cellrank_pancreas"
random_state = 0
max_cells = 6000
save_embedded = True
figure_dir = "figures"
default_preset = "balanced"
plot_pca_for_clasp_runs = False

available_presets()


In [ ]:
dataset = notebook_utils.dataset_config(selected_dataset)


def make_context(*, run_name, source, estimator, preprocess_params, graph_params):
    preprocess_overrides = {
        **preprocess_params,
        "max_cells": max_cells,
    }
    return {
        "selected_dataset": selected_dataset,
        "dataset": dataset,
        "optimized_params": {"source": source},
        "preprocess_overrides": preprocess_overrides,
        "estimator_params": {"n_components": estimator.n_components},
        "graph_params": graph_params,
        "estimator": estimator,
        "input_path": dataset["input_path"],
        "output_path": dataset["output_path"],
        "batch_key": dataset["batch_key"],
        "label_key": dataset["label_key"],
        "run_name": run_name,
    }


def run_case(context):
    adata, graph_params = notebook_utils.run_embedding_visualization(
        context,
        save=save_embedded,
        figure_dir=figure_dir,
        display_fn=display,
        close_figures=True,
        run_name=context["run_name"],
        plot_pca=plot_pca_for_clasp_runs,
    )
    summary = {
        "dataset": selected_dataset,
        "run": context["run_name"],
        "source": context["optimized_params"]["source"],
        "input_path": str(context["input_path"]),
        "embedded_path": context.get("embedded_path"),
        "batch_key": context["batch_key"],
        "label_key": context["label_key"],
        "preprocess": context["preprocess_overrides"],
        "estimator": context["estimator_params"],
        "graph": graph_params,
        "figures": context.get("figure_paths"),
    }
    return adata, pd.Series(summary, dtype="object")


dataset


## PCA Baseline

This plots the preprocessed PCA representation once before running CLASP variants.


In [ ]:
pca_estimator = ClaspEstimator(
    preset=default_preset,
    batch_key=dataset["batch_key"],
    label_key=dataset["label_key"],
    random_state=random_state,
)
pca_preprocess_params = {
    **pca_estimator.preprocess_defaults,
    **dataset.get("preprocess", {}),
    "max_cells": max_cells,
}
pca_adata = notebook_utils.load_preprocessed_data(
    pca_estimator,
    dataset,
    **pca_preprocess_params,
)
pca_path = f"{figure_dir}/{selected_dataset}_pca_baseline_embedding.pdf"
pca_axes = pca_estimator.plot(
    pca_adata,
    embedding_key="X_pca",
    filename=pca_path,
)
pca_fig = pca_axes[0].figure
display(pca_fig)
plt.close(pca_fig)

pd.Series({
    "dataset": selected_dataset,
    "run": "pca_baseline",
    "figure": pca_path,
    "preprocess": pca_preprocess_params,
}, dtype="object")


## Default Preset

This uses the named `balanced` preset unless `default_preset` is changed above.


In [ ]:
default_estimator = ClaspEstimator(
    preset=default_preset,
    batch_key=dataset["batch_key"],
    label_key=dataset["label_key"],
    random_state=random_state,
)
default_preprocess_params = {
    **default_estimator.preprocess_defaults,
    **dataset.get("preprocess", {}),
}
default_graph_params = {
    **default_estimator.graph_defaults,
    **dataset.get("graph", {}),
}
default_context = make_context(
    run_name=f"default_{default_preset}",
    source=f"preset:{default_preset}",
    estimator=default_estimator,
    preprocess_params=default_preprocess_params,
    graph_params=default_graph_params,
)

default_adata, default_summary = run_case(default_context)
default_summary


## Optimized Parameters

This cell uses `data/optimized_params/<dataset>_graph_params.json` when present. If no compatible file exists, it skips cleanly.


In [ ]:
try:
    optimized_payload = notebook_utils.load_optimized_graph_params(selected_dataset)
except FileNotFoundError as exc:
    optimized_payload = None
    print(exc)

if optimized_payload is not None:
    metadata = optimized_payload.get("metadata", {})
    compatible = (
        metadata.get("batch_key") == dataset["batch_key"]
        and metadata.get("label_key") == dataset["label_key"]
    )
    if not compatible:
        print("Optimized parameter file is stale for this dataset schema; rerun notebook 01 first.")
        optimized_summary = pd.Series({"run": "optimized", "status": "stale_optimized_params"})
    else:
        optimized_estimator = ClaspEstimator(
            batch_key=dataset["batch_key"],
            label_key=dataset["label_key"],
            random_state=random_state,
            **optimized_payload.get("estimator_params", {}),
        )
        optimized_preprocess_params = {
            **dataset.get("preprocess", {}),
            **optimized_payload.get("preprocess_params", {}),
        }
        optimized_context = make_context(
            run_name="optimized",
            source=str(notebook_utils.optimized_params_path(selected_dataset)),
            estimator=optimized_estimator,
            preprocess_params=optimized_preprocess_params,
            graph_params=optimized_payload["graph_params"],
        )
        optimized_adata, optimized_summary = run_case(optimized_context)
else:
    optimized_summary = pd.Series({"run": "optimized", "status": "missing_optimized_params"})

optimized_summary


## Editable Parameter Run

Change any value in this cell and rerun it to inspect the resulting embedding.


In [ ]:
custom_preprocess_defaults = {
    **get_preset(default_preset)["preprocess"],
    **dataset.get("preprocess", {}),
}
custom_preprocess_params = {
    "n_top_genes": custom_preprocess_defaults.get("n_top_genes", 1300),
    "min_cell_genes": custom_preprocess_defaults.get("min_cell_genes", None),
    "min_gene_counts": custom_preprocess_defaults.get("min_gene_counts", 0),
    "normalize": custom_preprocess_defaults.get("normalize", False),
    "hvg_flavor": custom_preprocess_defaults.get("hvg_flavor", "variance"),
    "create_artificial_batch": custom_preprocess_defaults.get("create_artificial_batch", False),
}

custom_estimator_params = {
    "n_components": get_preset(default_preset)["estimator"]["n_components"],
}

custom_graph_defaults = {
    **get_preset(default_preset)["graph"],
    **dataset.get("graph", {}),
}
custom_graph_params = {
    "n_neighbors": custom_graph_defaults["n_neighbors"],
    "intra_fraction": custom_graph_defaults["intra_fraction"],
    "n_inter_edges": custom_graph_defaults["n_inter_edges"],
    "metric": custom_graph_defaults["metric"],
    "assignment_quantile": custom_graph_defaults["assignment_quantile"],
    "hubness_correction": custom_graph_defaults["hubness_correction"],
    "hubness_k": custom_graph_defaults["hubness_k"],
    "rank_correction": custom_graph_defaults["rank_correction"],
    "edge_weighting": custom_graph_defaults["edge_weighting"],
    "inter_edge_mode": custom_graph_defaults["inter_edge_mode"],
    "mutual_neighbors": custom_graph_defaults["mutual_neighbors"],
    "neighbor_mode": custom_graph_defaults["neighbor_mode"],
    "symmetrize": custom_graph_defaults["symmetrize"],
}

custom_estimator = ClaspEstimator(
    batch_key=dataset["batch_key"],
    label_key=dataset["label_key"],
    random_state=random_state,
    **custom_estimator_params,
)
custom_context = make_context(
    run_name="custom",
    source="manual_cell",
    estimator=custom_estimator,
    preprocess_params=custom_preprocess_params,
    graph_params=custom_graph_params,
)

custom_adata, custom_summary = run_case(custom_context)
custom_summary
